# 05. 에이전트 구축 — Graph API와 Functional API로 ReAct 에이전트 만들기

## 학습 목표

도구를 사용하는 LLM 에이전트를 두 가지 API로 구현합니다.

- **Graph API**: `StateGraph`와 조건부 엣지로 ReAct 루프를 명시적으로 구성
- **Functional API**: `@entrypoint` + `while` 루프로 간결하게 구현
- **도구 바인딩**: `@tool` 데코레이터와 `bind_tools()`로 LLM에 도구 연결
- **메모리**: 체크포인터로 대화 상태를 유지하는 에이전트

## 5.1 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4.1")

In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 5.2 도구 정의 — @tool 데코레이터와 bind_tools()

LangChain의 `@tool` 데코레이터를 사용하면 일반 Python 함수를 LLM이 호출할 수 있는 도구로 바꿀 수 있습니다.
`bind_tools()`는 이 도구들의 스키마를 모델에 바인딩하여, LLM이 적절한 시점에 도구를 선택하고 인자를 생성하게 합니다.

In [3]:
from langchain.tools import tool
from langchain.messages import HumanMessage, SystemMessage, ToolMessage, AnyMessage

@tool
def add(a: int, b: int) -> int:
    """두 수를 더합니다."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """두 수를 곱합니다."""
    return a * b

@tool
def divide(a: int, b: int) -> float:
    """a를 b로 나눕니다."""
    return a / b

tools = [add, multiply, divide]
tools_by_name = {t.name: t for t in tools}
model_with_tools = model.bind_tools(tools)

print("모델에 바인딩된 도구:")
for t in tools:
    print(f"  - {t.name}: {t.description}")

모델에 바인딩된 도구:
  - add: 두 수를 더합니다.
  - multiply: 두 수를 곱합니다.
  - divide: a를 b로 나눕니다.


## 5.3 Graph API 에이전트 — StateGraph로 ReAct 루프 구현

ReAct(Reasoning + Acting) 패턴은 세 가지 요소로 구성됩니다:

- **LLM 노드**: 현재 메시지를 보고 도구 호출 여부를 결정합니다
- **Tool 노드**: LLM이 선택한 도구를 실제로 실행합니다
- **조건부 엣지**: `tool_calls`가 있으면 `tool_node`로, 없으면 `END`로 라우팅합니다

```
START → llm → [tool_calls?] → tools → llm → ... → END
```

In [4]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated, Literal
import operator

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

SYSTEM_PROMPT = "당신은 유용한 수학 어시스턴트입니다. 제공된 도구를 사용하여 계산하세요."

def llm_node(state: AgentState) -> dict:
    response = model_with_tools.invoke(
        [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"],
        config=lf_config,
)
    return {"messages": [response]}

def tool_node(state: AgentState) -> dict:
    results = []
    for tc in state["messages"][-1].tool_calls:
        tool_fn = tools_by_name[tc["name"]]
        result = tool_fn.invoke(tc["args"], config=lf_config)
        results.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
    return {"messages": results}

def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    last = state["messages"][-1]
    return "tools" if last.tool_calls else END

builder = StateGraph(AgentState)
builder.add_node("llm", llm_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", should_continue, ["tools", END])
builder.add_edge("tools", "llm")

agent = builder.compile()

result = agent.invoke({"messages": [HumanMessage(content="(7 * 8) + 12는 얼마인가요?")]}, config=lf_config)
print("에이전트 답변:", result["messages"][-1].content)

에이전트 답변: (7 × 8) + 12는 68입니다.


## 5.4 실행 흐름 시각화 — 스트리밍으로 각 단계 관찰

`stream_mode="updates"`를 사용하면 각 노드가 실행될 때마다 업데이트를 받을 수 있습니다.
에이전트가 어떤 순서로 도구를 호출하고 결과를 처리하는지 단계별로 볼 수 있습니다.

In [5]:
print("에이전트 실행 흐름:")
print("=" * 60)
for chunk in agent.stream(
    {"messages": [HumanMessage(content="100 / 4를 계산한 다음 3을 곱하면 얼마인가요?")]},
    stream_mode="updates",
    config=lf_config,
):
    for node_name, update in chunk.items():
        print(f"\n[{node_name}]")
        for msg in update.get("messages", []):
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  도구 호출: {tc['name']}({tc['args']})")
            elif hasattr(msg, 'content') and msg.content:
                print(f"  {msg.content[:200]}")

에이전트 실행 흐름:



[llm]
  도구 호출: divide({'a': 100, 'b': 4})

[tools]
  25.0



[llm]
  도구 호출: multiply({'a': 25, 'b': 3})

[tools]
  75



[llm]
  100을 4로 나누면 25가 되고, 여기에 3을 곱하면 75입니다. 정답은 75입니다.


## 5.5 Functional API 에이전트 — @entrypoint + while 루프

Functional API는 그래프를 명시적으로 구성하지 않고, 일반 Python 코드처럼 에이전트를 작성합니다.

- `@entrypoint`: 에이전트의 진입점을 정의합니다
- `@task`: 개별 작업 단위를 정의합니다
- `while` 루프: 도구 호출이 없을 때까지 반복합니다

In [6]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import add_messages

@task
def call_model(messages: list) -> object:
    return model_with_tools.invoke(
        [SystemMessage(content=SYSTEM_PROMPT)] + messages,
        config=lf_config,
)

@task
def execute_tool(tool_call: dict) -> ToolMessage:
    tool_fn = tools_by_name[tool_call["name"]]
    result = tool_fn.invoke(tool_call["args"], config=lf_config)
    return ToolMessage(content=str(result), tool_call_id=tool_call["id"])

@entrypoint(checkpointer=InMemorySaver())
def functional_agent(messages: list) -> list:
    response = call_model(messages).result()
    
    while response.tool_calls:
        tool_results = [execute_tool(tc).result() for tc in response.tool_calls]
        messages = add_messages(messages, [response] + tool_results)
        response = call_model(messages).result()
    
    return add_messages(messages, response)

config = {"configurable": {"thread_id": "func-agent-1"}}
result = functional_agent.invoke(
    [HumanMessage(content="15 * 4를 계산한 다음 20을 더하면 얼마인가요?")],
    {**config, **lf_config}
)
print("Functional 에이전트 답변:", result[-1].content)

Functional 에이전트 답변: 15 × 4 = 60이고, 여기에 20을 더하면 80입니다. 답은 80입니다.


## 5.6 메모리가 있는 에이전트 — 체크포인터로 대화 유지

체크포인터(`InMemorySaver`)를 `compile()`에 전달하면 에이전트가 이전 대화를 기억합니다.
같은 `thread_id`를 사용하면 이전 대화 컨텍스트가 자동으로 유지됩니다.

In [7]:
from langgraph.checkpoint.memory import InMemorySaver

agent_with_memory = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "math-session"}}

r1 = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="6 * 7은 얼마인가요?")]},
    {**config, **lf_config}
)
print("턴 1:", r1["messages"][-1].content)

r2 = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="이제 그 결과를 2로 나누세요.")]},
    {**config, **lf_config}
)
print("턴 2:", r2["messages"][-1].content)

턴 1: 6 × 7의 값은 42입니다.


턴 2: 42를 2로 나누면 21입니다.


## 5.7 요약

| 개념 | 설명 |
|------|------|
| `@tool` | Python 함수를 LLM 호출 가능한 도구로 변환 |
| `bind_tools()` | 도구 스키마를 모델에 바인딩 |
| **Graph API 에이전트** | `StateGraph` + 조건부 엣지로 ReAct 루프 명시적 구현 |
| **Functional API 에이전트** | `@entrypoint` + `while` 루프로 간결하게 구현 |
| `tool_calls` | LLM 응답에 포함된 도구 호출 정보 |
| `ToolMessage` | 도구 실행 결과를 LLM에게 전달하는 메시지 |
| **체크포인터** | 대화 상태를 저장하여 멀티턴 에이전트 구현 |

### 다음 단계
→ **[06_persistence_and_memory.ipynb](./06_persistence_and_memory.ipynb)**: 지속성과 메모리를 배웁니다.
